# ROCLING 2026 DSA — 情感知識圖譜生成增強：種子策略 × 風格錨定消融

論文重心實驗。六個合成資料條件 × 3 seed = **18 個 run**，A100 約 4.5 小時。
所有合成資料（各 400 篇）已 commit 進 repo，cell 2 `git clone` 直接拿得到。

| 條件 | seed_mode | 風格錨定 | 長度指令 | 檔案 | 回答什麼 |
|---|---|---|---|---|---|
| **N** | 不給種子詞 | ✗ | fixed | `train_aug_N.csv` | 種子詞有沒有用 |
| **A** | 隨機抽詞 | ✗ | fixed | `train_aug_A.csv` | VA 過濾 vs 隨機 |
| **C** | VA 查表（=實驗5） | ✗ | fixed | `train_aug_C.csv` | 基準線 |
| **E** | 圖擴散 G1+G3 | ✗ | fixed | `train_aug_E.csv` | **圖結構有沒有用** |
| **F** | 圖擴散 | ✓ Val | fixed | `train_aug_F.csv` | 風格錨定有沒有用 |
| **F2** | 圖擴散 | ✓ Val | match_real | `train_aug_F2.csv` | 放寬長度後錨定的效果 |

**核心對照**：C vs E（圖結構）、E vs F（錨定）、F vs F2（長度分布）。
**無增強基準線**免訓練：`macbert_s42/s1/s2` 已存在，dev A_PCC≈0.61。

⚠️ 評估紀律（見 `docs/experiments.md` §0：E4/E19 的 val→test 排序翻轉教訓）：
- 每條件 **3 seed**，比的是條件間 mean 差距 vs 條件內 seed std。
- **不要看單一 run 的分數就下結論**；dev 對增強實驗會失真，最終看官方 test。
- 嚴格 batch 32 / lr 2e-5 / 4 epochs，與實驗 4 對齊。


In [1]:
# 1) 安裝套件 + 確認 GPU
!pip -q install "transformers>=4.40" jieba scikit-learn scipy

import torch, subprocess
print('=' * 60)
if not torch.cuda.is_available():
    print('⚠️  沒有 GPU！請右上角 Select Kernel → Colab → 選 premium GPU runtime')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU：{name}（{vram:.1f} GB）')
print('=' * 60)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


✅ GPU：NVIDIA A100-SXM4-40GB（39.5 GB）
Thu Jul 23 13:48:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [3]:
# 2) git clone / pull 程式到 Colab runtime（token 用 getpass 輸入，不要寫進 notebook！）
import os, subprocess
from getpass import getpass

BRANCH = 'feat/ensemble-teacher-student-experiments'   # ← 程式所在 branch
REPO_PATH = '/content/repo' if os.path.exists('/content') else 'repo'

if not os.path.exists(REPO_PATH):
    token = getpass('GitHub token（public repo 直接按 Enter）: ').strip()
    prefix = f'{token}@' if token else ''
    !git clone -q -b {BRANCH} https://{prefix}github.com/chen0427ok/DSA-NIFT.git {REPO_PATH}
os.chdir(REPO_PATH)

# repo 已存在（舊 clone）時：切到正確 branch 並拉最新 commit
!git checkout -q {BRANCH}
r = subprocess.run(['git', 'pull', 'origin', BRANCH], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
if r.returncode != 0:
    token = getpass('git pull 失敗，輸入新的 GitHub token: ').strip()
    !git remote set-url origin https://{token}@github.com/chen0427ok/DSA-NIFT.git
    !git pull origin {BRANCH}

os.makedirs('outputs/preds', exist_ok=True)
print('工作目錄:', os.getcwd(), '| HEAD:',
      subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
assert os.path.exists('train_v2.py'), '沒拉到程式，檢查 branch / token'
# 六批合成資料都應該在 repo 裡（已 commit，非 gitignore）
import glob
augs = sorted(glob.glob('data/train_aug_[NACEF]*.csv'))
print('找到的增強檔:', [os.path.basename(a) for a in augs])
need = ['data/train_aug_%s.csv' % c for c in ['N','A','C','E','F','F2']]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, '缺增強檔: %s → git pull 沒成功？' % missing
print('✅ 六批合成資料就緒')


Updating b0a9146..ccc1eda
Fast-forward
 .gitignore                                         |   1 +
 CLAUDE.md                                          |  23 +
 README.md                                          | 245 ++----
 Rocling2026_Colab_e13.ipynb                        | 148 ----
 Rocling2026_Colab_e18_e21.ipynb                    | 307 -------
 Rocling2026_Colab_ensemble.ipynb                   | 190 -----
 affective_graph.py                                 |  99 +++
 analyze_variance.py                                | 119 +++
 augment_generate.py                                | 234 +++++-
 data/train_aug_A.csv                               | 401 +++++++++
 data/train_aug_C.csv                               | 401 +++++++++
 data/train_aug_E.csv                               | 401 +++++++++
 data/train_aug_F.csv                               | 401 +++++++++
 data/train_aug_F2.csv                              | 401 +++++++++
 data/train_aug_N.csv                               | 

## 3) 跑 18 個 run（六條件 × 3 seed）

`train_v2.py --extra_train data/train_aug_X.csv` 會把該批 400 篇併入訓練。
其餘參數全部預設 = 復現實驗 4（L1 詞典融合，batch 32 / lr 2e-5 / 4 epochs）。
每個 run 產出 `outputs/{run}_best.pt`、`outputs/preds/{run}_{dev,val}.csv`、`outputs/{run}_submission.csv`。

**已存在的 run 會自動跳過**（斷線續跑安全）。A100 每個 run 約 15 分鐘。


In [4]:
# 3) 主迴圈：六條件 × 3 seed
import os, subprocess, time

CONDITIONS = ['N', 'A', 'C', 'E', 'F', 'F2']
SEEDS = [42, 1, 2]

done, skipped = [], []
t0 = time.time()
for cond in CONDITIONS:
    for s in SEEDS:
        run = f'aug_{cond}_s{s}'
        if os.path.exists(f'outputs/{run}_submission.csv'):
            skipped.append(run); print(f'⏭  跳過 {run}（已存在）'); continue
        aug = f'data/train_aug_{cond}.csv'
        print(f'\n{"="*60}\n▶ {run}  ({aug})\n{"="*60}', flush=True)
        rc = subprocess.call([
            'python', 'train_v2.py',
            '--extra_train', aug,
            '--seed', str(s),
            '--run_name', run,
            '--epochs', '4', '--batch_size', '32', '--lr', '2e-5',
        ])
        if rc == 0:
            done.append(run)
        else:
            print(f'❌ {run} 失敗（rc={rc}），繼續下一個')

print(f'\n完成 {len(done)}／跳過 {len(skipped)}，耗時 {(time.time()-t0)/60:.0f} 分')
print('done:', done)



▶ aug_N_s42  (data/train_aug_N.csv)

▶ aug_N_s1  (data/train_aug_N.csv)

▶ aug_N_s2  (data/train_aug_N.csv)

▶ aug_A_s42  (data/train_aug_A.csv)

▶ aug_A_s1  (data/train_aug_A.csv)

▶ aug_A_s2  (data/train_aug_A.csv)

▶ aug_C_s42  (data/train_aug_C.csv)

▶ aug_C_s1  (data/train_aug_C.csv)

▶ aug_C_s2  (data/train_aug_C.csv)

▶ aug_E_s42  (data/train_aug_E.csv)

▶ aug_E_s1  (data/train_aug_E.csv)

▶ aug_E_s2  (data/train_aug_E.csv)

▶ aug_F_s42  (data/train_aug_F.csv)

▶ aug_F_s1  (data/train_aug_F.csv)

▶ aug_F_s2  (data/train_aug_F.csv)

▶ aug_F2_s42  (data/train_aug_F2.csv)

▶ aug_F2_s1  (data/train_aug_F2.csv)

▶ aug_F2_s2  (data/train_aug_F2.csv)

完成 18／跳過 0，耗時 122 分
done: ['aug_N_s42', 'aug_N_s1', 'aug_N_s2', 'aug_A_s42', 'aug_A_s1', 'aug_A_s2', 'aug_C_s42', 'aug_C_s1', 'aug_C_s2', 'aug_E_s42', 'aug_E_s1', 'aug_E_s2', 'aug_F_s42', 'aug_F_s1', 'aug_F_s2', 'aug_F2_s42', 'aug_F2_s1', 'aug_F2_s2']


## 4) 彙整 dev 分數（快速看趨勢；⚠️ dev 對增強實驗會失真，最終仍看官方 test）

每條件的 3 seed 算 mean ± std。**要看的是條件間 mean 差距是否大於條件內 std**，
不是單一 run 的分數。


In [5]:
# 4) dev 四指標 mean±std（六條件）
import glob, os, numpy as np, pandas as pd

def metrics(df):
    out = {}
    for j in ['valence', 'arousal']:
        p, g = df[f'{j}_pred'].values, df[f'{j}_true'].values
        out[f'{j[0].upper()}_MAE'] = np.mean(np.abs(p - g))
        out[f'{j[0].upper()}_PCC'] = np.corrcoef(p, g)[0, 1] if p.std() > 1e-8 else 0.0
    return out

rows = []
for cond in ['N', 'A', 'C', 'E', 'F', 'F2']:
    per = [metrics(pd.read_csv(f)) for f in sorted(glob.glob(f'outputs/preds/aug_{cond}_s*_dev.csv'))]
    if not per:
        continue
    dfm = pd.DataFrame(per)
    rec = {'cond': cond, 'n_seed': len(per)}
    for k in ['V_MAE', 'V_PCC', 'A_MAE', 'A_PCC']:
        rec[k] = f'{dfm[k].mean():.3f}±{dfm[k].std():.3f}'
    rows.append(rec)

print(pd.DataFrame(rows).to_string(index=False))
print('\n對照：無增強基準（macbert_s42/s1/s2）也可用同法算，作為所有條件的 baseline。')
print('⚠️ dev 分數僅供趨勢；C vs E vs F2 的結論必須以官方 test 提交為準。')


cond  n_seed       V_MAE       V_PCC       A_MAE       A_PCC
   N       3 0.488±0.012 0.810±0.006 0.844±0.010 0.603±0.009
   A       3 0.493±0.013 0.809±0.005 0.848±0.013 0.601±0.012
   C       3 0.488±0.010 0.808±0.005 0.848±0.009 0.598±0.009
   E       3 0.488±0.013 0.810±0.007 0.847±0.016 0.599±0.013
   F       3 0.496±0.017 0.806±0.009 0.850±0.008 0.600±0.005
  F2       3 0.488±0.016 0.810±0.011 0.850±0.004 0.602±0.006

對照：無增強基準（macbert_s42/s1/s2）也可用同法算，作為所有條件的 baseline。
⚠️ dev 分數僅供趨勢；C vs E vs F2 的結論必須以官方 test 提交為準。


## ⭐ 訓練完成後：在 Colab 直接產生 18 個 test submission（推薦，免搬 6GB 權重）

`kg_ablation_results.zip` 有 6.37 GB（18 顆權重），透過 Drive/下載都很痛。
**你不需要那些權重**——真正要的是能上傳 leaderboard 的 test submission（每個幾 KB）。
本 cell 用 repo 內的 checkpoint 對官方 test set 推論，產生 18 個 submission，
再打包成一個小 zip，並（可選）用 git 推回 repo，本地 `git pull` 就拿得到。


In [ ]:
# 6) 對官方 test set 產生 18 個 submission（小檔）
import os, glob, subprocess

TEST_CSV = 'data/DSANIDF_TestSet.csv'          # 已 commit 進 repo
assert os.path.exists(TEST_CSV), '缺 test set → 先在本機 git pull 拉最新 commit'

runs = [f'aug_{c}_s{s}' for c in ['N','A','C','E','F','F2'] for s in [42,1,2]]
for run in runs:
    ckpt = f'outputs/{run}_best.pt'
    if not os.path.exists(ckpt):
        print(f'⚠ 缺 {ckpt}（該 run 沒訓練完？）'); continue
    if os.path.exists(f'outputs/{run}_test_submission.csv'):
        print(f'⏭ {run} 已有 test submission'); continue
    print(f'▶ predict {run}', flush=True)
    subprocess.call(['python','predict.py','--ckpt',ckpt,'--lex_mode','l1',
                     '--input',TEST_CSV,'--run_name',run,'--split','test',
                     '--batch_size','32'])

# 打包小檔：submission + 所有 dev/val/test 預測（幾 MB，非 6GB）
import shutil
os.makedirs('kg_small', exist_ok=True)
for f in glob.glob('outputs/*_submission.csv') + glob.glob('outputs/preds/*.csv'):
    shutil.copy(f, 'kg_small/')
zp = shutil.make_archive('kg_ablation_small', 'zip', 'kg_small')
print('\n✅ 小結果包 ->', os.path.abspath(zp), f'({os.path.getsize(zp)/1024**2:.1f} MB)')
print('   內含', len(glob.glob('kg_small/*')), '個檔')


## 7) 把小結果推回 repo（最省事的拉回方式）

submission / 預測都是小檔，直接 `git add -f` 推回 branch，本地 `git pull` 即可。
（token 用 getpass；若 cell 2 的 remote 已含 token 可直接 push。）


In [ ]:
# 7) git push 小結果回 repo（本地 git pull 拿回）
import subprocess
# Colab VM 沒有 git 身分 → 先設一個（commit 失敗會導致「Everything up-to-date」）
!git config user.email "colab@dsanift.local"
!git config user.name "colab"
!git add -f outputs/*_test_submission.csv outputs/preds/aug_*.csv
r = subprocess.run(['git','commit','-m','KG 消融 18 run 的 test/dev/val 預測與 submission'],
                   capture_output=True, text=True)
print((r.stdout+r.stderr).strip())
p = subprocess.run(['git','push','origin',BRANCH], capture_output=True, text=True)
print((p.stdout+p.stderr).strip() or '✅ pushed')


## 8)（可選）真的要保留 6GB 權重才跑：複製到 Google Drive

只有在你想留 18 顆 `.pt`（之後還要重推論）時才需要。否則跳過——
論文只需要 cell 6/7 的小檔。這也是原 cell「Drive 沒出現壓縮檔」的正確做法：
`files.download()` 在 VS Code Colab 模式不會動，必須用 drive.mount + copy。


In [12]:
# 8)（可選）掛 Drive 並複製大權重包
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
big = 'kg_ablation_results.zip'
if not os.path.exists(big):
    big = shutil.make_archive('kg_ablation_results', 'zip', 'outputs')
dst = '/content/drive/MyDrive/kg_ablation_results.zip'
shutil.copy(big, dst)
print('✅ 已複製到', dst, f'({os.path.getsize(dst)/1024**3:.2f} GB)')


Mounted at /content/drive
✅ 已複製到 /content/drive/MyDrive/kg_ablation_results.zip (6.37 GB)


## 9) no-L1 消融（釘死「L1 詞典融合是否為真正的正面貢獻」）

方向 A 需要知道 L1 在 **test** 上到底有沒有用（val 上有用，但 val 已證不可信）。
訓練 3 顆 `--lex_mode none`（其餘與 E4 相同），推論 test，推回 repo。
與無增強 baseline（macbert_s*，有 L1）對照：若 no-L1 的 test A_PCC 明顯較低 → L1 是正面貢獻；
若一樣 → 連 L1 都是 val 幻覺（方向 A 的故事更純粹）。


In [ ]:
# 9) no-L1：訓練 3 seed + test 推論 + push
import os, subprocess
!git config user.email "colab@dsanift.local" 2>/dev/null; git config user.name "colab" 2>/dev/null
for s in [42,1,2]:
    run=f'nolex_s{s}'
    if not os.path.exists(f'outputs/{run}_best.pt'):
        print(f'▶ train {run}', flush=True)
        subprocess.call(['python','train_v2.py','--lex_mode','none','--seed',str(s),
                         '--run_name',run,'--epochs','4','--batch_size','32','--lr','2e-5'])
    if not os.path.exists(f'outputs/{run}_test_submission.csv'):
        subprocess.call(['python','predict.py','--ckpt',f'outputs/{run}_best.pt',
                         '--lex_mode','none','--input','data/DSANIDF_TestSet.csv',
                         '--run_name',run,'--split','test','--batch_size','32'])
!git add -f outputs/nolex_*_test_submission.csv outputs/preds/nolex_*.csv outputs/nolex_*_best.pt 2>/dev/null
!git add -f outputs/nolex_*_test_submission.csv outputs/preds/nolex_*.csv
r=subprocess.run(['git','commit','-m','no-L1 消融 3 seed：test 預測與 submission'],capture_output=True,text=True)
print((r.stdout+r.stderr).strip())
p=subprocess.run(['git','push','origin',BRANCH],capture_output=True,text=True)
print((p.stdout+p.stderr).strip() or 'pushed')
